# Week 5 — Feature Extraction & Tokenizers

This notebook shows how to prepare audio inputs for popular ASR models: Wav2Vec2 (CTC) and Whisper (encoder-decoder). We load a tiny audio sample produced earlier, then run the processors to produce model-ready inputs.
Designed for Colab — installs are included.

In [1]:
# Install lightweight dependencies
!pip install -q transformers datasets soundfile librosa

In [5]:
import numpy as np
from datasets import load_dataset, Audio
import soundfile as sf
import librosa
from transformers import Wav2Vec2Processor, WhisperProcessor

# Load a tiny sample (same fallback logic as notebook 1)
try:
    ds = load_dataset('mozilla-foundation/common_voice_13_0', 'en', split='train[:0.1%]')
    ds_name = 'common_voice_13_0'
except Exception as e:
    print('Common Voice not available, using librispeech demo:', e)
    try:
        ds = load_dataset('hf-internal-testing/librispeech_asr_demo', split='validation')
        ds_name = 'librispeech_asr_demo'
    except Exception as e2:
        print('Fallback librispeech demo failed:', e2)
        # Final fallback: synthetic 1s tone
        sr = 16000
        duration = 1.0
        t = np.linspace(0, duration, int(sr * duration), endpoint=False)
        arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')
        example = {'audio': {'array': arr, 'sampling_rate': sr}, 'text': 'synthetic tone'}
        ds = None
        ds_name = 'synthetic'

if ds_name != 'synthetic':
    ds = ds.cast_column('audio', Audio(decode=False))
    example = ds[0]
else:
    example = {'audio': {'array': arr, 'sampling_rate': 16000}, 'text': 'synthetic tone'}

# Extract audio safely
arr = None
sr = 16000
if 'audio' in example:
    audio = example['audio']
    if isinstance(audio, dict) and 'array' in audio:
        arr = np.array(audio['array']).astype('float32')
        sr = int(audio.get('sampling_rate', 16000))
    elif isinstance(audio, dict) and 'path' in audio:
        path = audio['path']
        try:
            arr, sr = sf.read(path)
            arr = np.array(arr).astype('float32')
        except Exception as e_path:
            print('soundfile path read failed; trying librosa:', e_path)
            try:
                arr, sr = librosa.load(path, sr=None)
                arr = np.array(arr).astype('float32')
            except Exception as e_lib:
                print('librosa path read failed; falling back to synthetic:', e_lib)
    else:
        try:
            arr, sr = sf.read(audio)
            arr = np.array(arr).astype('float32')
        except Exception:
            sr = 16000
            duration = 1.0
            t = np.linspace(0, duration, int(sr * duration), endpoint=False)
            arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')
else:
    sr = 16000
    duration = 1.0
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')

# Ensure float32 and resample to 16 kHz if needed
if arr is None:
    sr = 16000
    duration = 1.0
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')

if sr != 16000:
    arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    sr = 16000

print('Sample rate:', sr, 'duration s:', len(arr)/sr, 'source:', ds_name)

# Wav2Vec2 processor (feature extractor + tokenizer). We only use the processor part for features here.
print('Loading Wav2Vec2 processor (this downloads model assets but is needed for preprocessing)')
proc_w2v = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
inputs = proc_w2v(arr, sampling_rate=sr, return_tensors='pt', padding=True)
print('Wav2Vec2 input keys:', list(inputs.keys()))
print('Input input_values shape:', inputs['input_values'].shape)

# Whisper processor example (encoder-decoder style). Use a small pretrained whisper processor only for tokenization/feature prep
print('Loading Whisper processor (tokenizer + feature extractor)')
try:
    proc_whisper = WhisperProcessor.from_pretrained('openai/whisper-small')
    whisper_inputs = proc_whisper(arr, sampling_rate=sr, return_tensors='pt')
    print('Whisper input keys:', list(whisper_inputs.keys()))
except Exception as e:
    print('Could not load Whisper processor (likely heavy for Colab CPU). Skipping Whisper demo:', e)

print('Done — features prepared for a single sample.')


Repo card metadata block was not found. Setting CardData to empty.


Common Voice not available, using librispeech demo: The directory at hf://datasets/mozilla-foundation/common_voice_13_0@ff2bbb54dcdb597100fe534a1b911ff9103f9e22 doesn't contain any data files
soundfile path read failed; trying librosa: Error opening '1272-128104-0000.flac': System error.


/tmp/ipython-input-46473574.py:49: UserWarning: PySoundFile failed. Trying audioread instead.
  arr, sr = librosa.load(path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


librosa path read failed; falling back to synthetic: [Errno 2] No such file or directory: '1272-128104-0000.flac'
Sample rate: 16000 duration s: 1.0 source: librispeech_asr_demo
Loading Wav2Vec2 processor (this downloads model assets but is needed for preprocessing)


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Whisper input keys: ['input_features']
Done — features prepared for a single sample.
